# Correctness memory probe — $z_{\mathrm{combined}}$ vs magnitude vs $H$

Uses **artifacts from `dermamnist_correctness_memory_full.ipynb`** (no retraining in this notebook).

Correctness memory: $M^{(j)} \in \mathbb{R}^{7 \times d_k}$, $z_j \in \mathbb{R}^7$, $z_{\mathrm{combined}}=[z_1,\ldots,z_4]\in\mathbb{R}^{28}$.

| Phase | Question |
|-------|----------|
| **1** | Do full vectors $z_1,\ldots,z_4$ beat magnitudes $\|z_j\|$ for error detection? |
| **3** | Does unified $z_{\mathrm{combined}}$ (on $X_{\mathrm{best}}$) beat normalized entropy $\tilde H$? |

## Workflow

| Run once | Re-run when changing test size |
|----------|--------------------------------|
| **Configuration** → **Setup** → **Load / deploy features** | **Eval configuration** → **Phase 1** → **Phase 3** |

Feature CSVs are read from `correctness_feature_cache/` (written by `dermamnist_correctness_memory_full.ipynb` Part II). Set `FORCE_REDEPLOY=True` only if you need to regenerate from saved correctness NPZ artifacts.

## Configuration — paths & feature cache

Paths match `dermamnist_correctness_memory_full.ipynb`. Run that notebook's **Part I–II** first (or set `FORCE_REDEPLOY=True` here).

In [1]:
from pathlib import Path

TASK = "dermamnist"
SEEDS = (42, 123, 456)
CORRECTNESS_Z_DIM = 7  # z_j in R^7; z_combined in R^{28}
LOGISTIC_C = 1.0

# True → redeploy from correctness NPZ (slow). False → use correctness_feature_cache from dermamnist notebook.
FORCE_REDEPLOY = False

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "research").exists():
    REPO_ROOT = REPO_ROOT.parent

ARTIFACT_DIR = REPO_ROOT / "research" / "correctness_memory_fft" / "artifacts"
DATA_DIR = REPO_ROOT / "data" / "clinical"
PROBE_OUTPUT_DIR = REPO_ROOT / "research" / "correctness_memory_fft" / "probe_experiments"
FEATURE_CACHE_DIR = PROBE_OUTPUT_DIR / "correctness_feature_cache"
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
PROBE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT", REPO_ROOT)
print("ARTIFACT_DIR", ARTIFACT_DIR)
print("FEATURE_CACHE_DIR", FEATURE_CACHE_DIR)
print(f"CORRECTNESS_Z_DIM={CORRECTNESS_Z_DIM}  z_combined dim={CORRECTNESS_Z_DIM * 4}")
print("FORCE_REDEPLOY", FORCE_REDEPLOY)
print("SEEDS", SEEDS)

REPO_ROOT C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection
ARTIFACT_DIR C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\correctness_memory_fft\artifacts
FEATURE_CACHE_DIR C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\correctness_memory_fft\probe_experiments\correctness_feature_cache
CORRECTNESS_Z_DIM=7  z_combined dim=28
FORCE_REDEPLOY False
SEEDS (42, 123, 456)


## Setup

Load bundle and correctness-memory features ($z_j \in \mathbb{R}^7$). Labels joined only for error metric $e$.

In [2]:
from __future__ import annotations

import json
import pickle
import sys
from typing import Any

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.common.clinical_datasets import ClinicalDatasetConfig, load_clinical_bundle
from research.common.clinical_training import checkpoint_path
from research.common.correctness_deployment_pipeline import run_correctness_deployment_on_split
from research.common.correctness_memory_io import (
    correctness_artifacts_exist,
    load_correctness_artifacts,
)

MAG_COLS = ["z1_magnitude", "z2_magnitude", "z3_magnitude", "z4_magnitude"]
NUM_LEVELS = 4


def z_combined_cols(*, z_dim: int = CORRECTNESS_Z_DIM) -> list[str]:
    cols: list[str] = []
    for j in range(1, NUM_LEVELS + 1):
        cols.extend([f"z{j}_d{d}" for d in range(z_dim)])
    return cols


def infer_z_dim(df: pd.DataFrame) -> int:
    cols = [c for c in df.columns if c.startswith("z1_d")]
    if not cols:
        raise ValueError("Feature cache missing z1_d* columns; run dermamnist_correctness_memory_full Part II.")
    return len(cols)


Z_COMBINED_COLS = z_combined_cols()


def feature_cache_paths(seed: int) -> tuple[Path, Path]:
    seed_dir = FEATURE_CACHE_DIR / f"seed{seed}"
    return seed_dir / "cal_features.csv", seed_dir / "test_features_full.csv"


def subsample_test_df(test_full: pd.DataFrame, *, sample_size: int | None, seed: int) -> pd.DataFrame:
    if sample_size is None or sample_size >= len(test_full):
        return test_full.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(test_full), size=int(sample_size), replace=False))
    return test_full.iloc[idx].reset_index(drop=True)


def fit_l2_probe_on_cal(
    cal_df: pd.DataFrame,
    feature_cols: list[str],
    *,
    representation: str,
) -> dict[str, Any]:
    """Fit scaler + L2 logistic on calibration only."""
    req = feature_cols + ["error"]
    mask_cal = cal_df[req].notna().all(axis=1).to_numpy()
    x_cal = cal_df.loc[mask_cal, feature_cols].to_numpy(dtype=np.float64)
    y_cal = cal_df.loc[mask_cal, "error"].to_numpy(dtype=int)

    if len(x_cal) < 10:
        raise ValueError(f"Too few calibration samples for {representation}: n={len(x_cal)}")
    if len(np.unique(y_cal)) < 2:
        raise ValueError(f"Calibration errors are single-class for {representation}")

    pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(C=LOGISTIC_C, max_iter=5000, random_state=42),
            ),
        ]
    )
    pipe.fit(x_cal, y_cal)

    scaler = pipe.named_steps["scaler"]
    clf = pipe.named_steps["clf"]
    coef_scaled = clf.coef_.reshape(-1)
    coef_raw = coef_scaled / scaler.scale_

    coef_rows = [
        {
            "feature": name,
            "coef_standardized": float(w_s),
            "coef_raw_scale": float(w_r),
        }
        for name, w_s, w_r in zip(feature_cols, coef_scaled, coef_raw, strict=True)
    ]

    return {
        "representation": representation,
        "feature_cols": feature_cols,
        "n_cal": int(len(x_cal)),
        "intercept": float(clf.intercept_[0]),
        "coefficients": pd.DataFrame(coef_rows),
        "pipeline": pipe,
    }


def eval_l2_probe(
    fitted: dict[str, Any],
    test_df: pd.DataFrame,
) -> dict[str, Any]:
    """Evaluate a calibration-fitted probe on a (possibly subsampled) test set."""
    feature_cols = fitted["feature_cols"]
    req = feature_cols + ["error"]
    mask_test = test_df[req].notna().all(axis=1).to_numpy()
    x_test = test_df.loc[mask_test, feature_cols].to_numpy(dtype=np.float64)
    y_test = test_df.loc[mask_test, "error"].to_numpy(dtype=int)

    if len(x_test) < 10:
        raise ValueError(f"Too few test samples: n={len(x_test)}")
    if len(np.unique(y_test)) < 2:
        raise ValueError("Test errors are single-class for this subset")

    pipe = fitted["pipeline"]
    scores = pipe.predict_proba(x_test)[:, 1]
    return {
        **fitted,
        "n_test": int(len(x_test)),
        "auroc": float(roc_auc_score(y_test, scores)),
        "auprc": float(average_precision_score(y_test, scores)),
        "test_scores": scores,
        "test_errors": y_test,
    }


def fit_and_eval_probe(
    cal_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    *,
    representation: str,
) -> dict[str, Any]:
    fitted = fit_l2_probe_on_cal(cal_df, feature_cols, representation=representation)
    return eval_l2_probe(fitted, test_df)


def eval_entropy_baseline(test_df: pd.DataFrame) -> dict[str, float]:
    mask = test_df[["normalized_entropy", "error"]].notna().all(axis=1).to_numpy()
    y = test_df.loc[mask, "error"].to_numpy(dtype=int)
    h = test_df.loc[mask, "normalized_entropy"].to_numpy(dtype=float)
    if len(y) < 10 or len(np.unique(y)) < 2:
        return {"n_test": int(len(y)), "auroc": float("nan"), "auprc": float("nan")}
    return {
        "n_test": int(len(y)),
        "auroc": float(roc_auc_score(y, h)),
        "auprc": float(average_precision_score(y, h)),
    }


def deploy_split_for_seed(
    seed: int,
    x: np.ndarray,
    y: np.ndarray,
    ids: np.ndarray,
) -> pd.DataFrame:
    if not correctness_artifacts_exist(ARTIFACT_DIR, TASK, seed):
        raise FileNotFoundError(
            f"Missing correctness artifacts for seed={seed} under {ARTIFACT_DIR}. "
            "Run dermamnist_correctness_memory_full.ipynb Part I first."
        )
    with open(checkpoint_path(ARTIFACT_DIR, TASK, seed), "rb") as f:
        params = pickle.load(f)["params"]
    mem_state, _, _ = load_correctness_artifacts(ARTIFACT_DIR, TASK, seed)
    _, df = run_correctness_deployment_on_split(
        params,
        x,
        y,
        ids,
        mem_state,
        num_classes=bundle.num_classes,
    )
    return df


def load_or_deploy_features(seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    cal_path, test_path = feature_cache_paths(seed)
    if not FORCE_REDEPLOY and cal_path.exists() and test_path.exists():
        cal_df = pd.read_csv(cal_path)
        test_df = pd.read_csv(test_path)
        if infer_z_dim(cal_df) == CORRECTNESS_Z_DIM:
            return cal_df, test_df
        print(f"  stale cache seed={seed} (z_dim={infer_z_dim(cal_df)}); redeploying")

    print(f"  deploying seed={seed} (full cal + full test) ...", flush=True)
    cal_df = deploy_split_for_seed(
        seed,
        bundle.x_cal,
        bundle.y_cal,
        bundle.sample_ids["cal"],
    )
    test_full = deploy_split_for_seed(
        seed,
        bundle.x_test,
        bundle.y_test,
        bundle.sample_ids["test"],
    )
    cal_path.parent.mkdir(parents=True, exist_ok=True)
    cal_df.to_csv(cal_path, index=False)
    test_full.to_csv(test_path, index=False)
    return cal_df, test_full


bundle = load_clinical_bundle(ClinicalDatasetConfig(task=TASK, data_dir=DATA_DIR))
print(f"bundle: n_cal={len(bundle.x_cal)} n_test={len(bundle.x_test)} num_classes={bundle.num_classes}")

bundle: n_cal=1602 n_test=2005 num_classes=7


In [3]:
# Load full cal + full test features (same cache as dermamnist_correctness_memory_full Part II)

cal_frames_full: dict[int, pd.DataFrame] = {}
test_frames_full: dict[int, pd.DataFrame] = {}

for seed in SEEDS:
    cal_df, test_full = load_or_deploy_features(seed)
    cal_frames_full[seed] = cal_df
    test_frames_full[seed] = test_full
    z_dim = infer_z_dim(cal_df)
    print(
        f"seed={seed}: cal={len(cal_df)} test_full={len(test_full)} "
        f"z_dim={z_dim} redeploy={FORCE_REDEPLOY}"
    )

print("feature cache ready →", FEATURE_CACHE_DIR)

seed=42: cal=1602 test_full=2005 z_dim=7 redeploy=False
seed=123: cal=1602 test_full=2005 z_dim=7 redeploy=False
seed=456: cal=1602 test_full=2005 z_dim=7 redeploy=False
feature cache ready → C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\correctness_memory_fft\probe_experiments\correctness_feature_cache


## Eval configuration — change test size here

Re-run **this cell** and the **Phase 1 / Phase 3** cells below. No redeployment.

In [4]:
TEST_SAMPLE_SIZE = 2000  # None = full test set
RANDOM_SEED = 42

# Build eval subsets from cached full test features
cal_frames = cal_frames_full
test_frames = {
    seed: subsample_test_df(test_frames_full[seed], sample_size=TEST_SAMPLE_SIZE, seed=RANDOM_SEED)
    for seed in SEEDS
}

print(f"TEST_SAMPLE_SIZE={TEST_SAMPLE_SIZE}  RANDOM_SEED={RANDOM_SEED}")
for seed in SEEDS:
    te = test_frames[seed]
    print(f"  seed={seed}: n_test={len(te)} error_rate={te['error'].mean():.3f}")

TEST_SAMPLE_SIZE=2000  RANDOM_SEED=42
  seed=42: n_test=2000 error_rate=0.332
  seed=123: n_test=2000 error_rate=0.332
  seed=456: n_test=2000 error_rate=0.332


---

# Phase 1 — $\|z\|$ vs $z_{\mathrm{combined}}$

\[
X_{\text{mag}}=[\|z_1\|,\ldots,\|z_4\|],\quad
z_{\text{combined}}=[z_1,\ldots,z_4]\in\mathbb R^{28}
\]

Train separate L2 logistic probes on calibration; evaluate on the held-out test subset.
Select $X_{\text{best}}$ by **calibration AUROC** (mean across seeds).

In [5]:
phase1_rows: list[dict[str, Any]] = []
phase1_coefs: dict[int, dict[str, pd.DataFrame]] = {}
phase1_runs: dict[int, dict[str, Any]] = {}

for seed in SEEDS:
    cal_df = cal_frames[seed]
    test_df = test_frames[seed]
    mag_fit = fit_l2_probe_on_cal(cal_df, MAG_COLS, representation="magnitude")
    z_combined_fit = fit_l2_probe_on_cal(cal_df, Z_COMBINED_COLS, representation="z_combined")
    mag_cal = eval_l2_probe(mag_fit, cal_df)
    z_combined_cal = eval_l2_probe(z_combined_fit, cal_df)
    mag = eval_l2_probe(mag_fit, test_df)
    z_combined = eval_l2_probe(z_combined_fit, test_df)
    phase1_runs[seed] = {"magnitude": mag, "z_combined": z_combined}
    phase1_coefs[seed] = {
        "magnitude": mag["coefficients"],
        "z_combined": z_combined["coefficients"],
    }
    for run, cal_run in ((mag, mag_cal), (z_combined, z_combined_cal)):
        phase1_rows.append(
            {
                "seed": seed,
                "representation": run["representation"],
                "n_cal": run["n_cal"],
                "n_test": run["n_test"],
                "cal_auroc": cal_run["auroc"],
                "test_auroc": run["auroc"],
                "test_auprc": run["auprc"],
                "intercept": run["intercept"],
            }
        )

phase1_df = pd.DataFrame(phase1_rows)
phase1_summary = (
    phase1_df.groupby("representation")[["cal_auroc", "test_auroc", "test_auprc"]]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)

mag_cal_mean = float(phase1_df.loc[phase1_df["representation"] == "magnitude", "cal_auroc"].mean())
z_combined_cal_mean = float(phase1_df.loc[phase1_df["representation"] == "z_combined", "cal_auroc"].mean())
mag_test_mean = float(phase1_df.loc[phase1_df["representation"] == "magnitude", "test_auroc"].mean())
z_combined_test_mean = float(phase1_df.loc[phase1_df["representation"] == "z_combined", "test_auroc"].mean())
if z_combined_cal_mean >= mag_cal_mean:
    x_best_name = "z_combined"
    x_best_cols = Z_COMBINED_COLS
else:
    x_best_name = "magnitude"
    x_best_cols = MAG_COLS

phase1_decision = {
    "x_best": x_best_name,
    "memory_kind": "correctness_hd",
    "correctness_z_dim": CORRECTNESS_Z_DIM,
    "mean_cal_auroc_magnitude": mag_cal_mean,
    "mean_cal_auroc_z_combined": z_combined_cal_mean,
    "mean_test_auroc_magnitude": mag_test_mean,
    "mean_test_auroc_z_combined": z_combined_test_mean,
    "z_combined_beats_magnitude_on_cal_auroc": z_combined_cal_mean >= mag_cal_mean,
    "feature_dim": len(x_best_cols),
}

print("Phase 1 per-seed results")
display(phase1_df)
print("\nPhase 1 aggregate (mean ± std across seeds)")
display(phase1_summary)
print("\nPhase 1 decision:", phase1_decision)

for seed in SEEDS:
    print(f"\n--- seed={seed} magnitude coefficients ---")
    display(phase1_coefs[seed]["magnitude"])
    print(f"--- seed={seed} z_combined coefficients (first 10) ---")
    display(phase1_coefs[seed]["z_combined"].head(10))

Phase 1 per-seed results


,seed,representation,n_cal,n_test,cal_auroc,test_auroc,test_auprc,intercept
0,42,magnitude,1602,2000,0.726483,0.729885,0.533516,-0.822630
1,42,z_combined,1602,2000,0.788777,0.765281,0.628731,-0.900228
2,123,magnitude,1602,2000,0.680674,0.685911,0.487813,-0.769506
3,123,z_combined,1602,2000,0.782311,0.750404,0.611199,-0.868352
4,456,magnitude,1602,2000,0.714078,0.713340,0.526063,-0.792594
5,456,z_combined,1602,2000,0.765740,0.744384,0.582077,-0.862382



Phase 1 aggregate (mean ± std across seeds)


cal_auroc                         test_auroc                  \
                    mean     std     min     max       mean     std     min   
representation                                                                
magnitude         0.7071  0.0237  0.6807  0.7265     0.7097  0.0222  0.6859   
z_combined              0.7789  0.0119  0.7657  0.7888     0.7534  0.0108  0.7444   

                       test_auprc                          
                   max       mean     std     min     max  
representation                                             
magnitude       0.7299     0.5158  0.0245  0.4878  0.5335  
z_combined            0.7653     0.6073  0.0236  0.5821  0.6287


Phase 1 decision: {'x_best': 'z_combined', 'memory_kind': 'correctness_hd', 'correctness_z_dim': 7, 'mean_cal_auroc_magnitude': 0.7070783057354989, 'mean_cal_auroc_z_combined': 0.7789424911516041, 'mean_test_auroc_magnitude': 0.7097123182740677, 'mean_test_auroc_z_combined': 0.7533562491985651, 'z_combined_beats_magnitude_on_cal_auroc': True, 'feature_dim': 28}

--- seed=42 magnitude coefficients ---


...

,feature,coef_standardized,coef_raw_scale
0,z1_d0,0.153416,224.020885
1,z1_d1,1.850349,2312.608742
2,z1_d2,-0.795427,-1175.742236
3,z1_d3,2.442849,4160.463577
4,z1_d4,-0.197260,-167.481171
5,z1_d5,0.123285,613.279311
6,z1_d6,1.525688,2663.812608
7,z2_d0,-0.333107,-2727.450094
8,z2_d1,0.410159,2174.073956
9,z2_d2,0.212327,1454.793453



--- seed=456 magnitude coefficients ---


,feature,coef_standardized,coef_raw_scale
0,z1_magnitude,-1.250995,-1453.088633
1,z2_magnitude,1.235934,5334.029419
2,z3_magnitude,1.651050,25330.496922
3,z4_magnitude,-2.284844,-253949.075633


--- seed=456 z_combined coefficients (first 10) ---


,feature,coef_standardized,coef_raw_scale
0,z1_d0,1.147628,5526.091544
1,z1_d1,2.639270,4722.803148
2,z1_d2,0.824487,2335.418371
3,z1_d3,1.038601,2553.235151
4,z1_d4,-1.369591,-3923.535540
5,z1_d5,-0.006057,-21.210533
6,z1_d6,0.206287,816.860005
7,z2_d0,-0.127854,-2659.862959
8,z2_d1,1.126949,7686.196054
9,z2_d2,0.709605,7642.652681


---

# Phase 3 — Correctness memory vs entropy

Using $X_{\text{best}}$ from Phase 1, train one probe on calibration:

\[
z_{\text{combined}}(x)=P(e=1\mid X_{\text{best}}), \qquad s_{\text{fail}}=1-q(x)
\]

Compare against normalized entropy baseline $\tilde H(x)$ on the same test subset.

In [6]:
phase3_rows: list[dict[str, Any]] = []
phase3_coefs: dict[int, pd.DataFrame] = {}

for seed in SEEDS:
    cal_df = cal_frames[seed]
    test_df = test_frames[seed]
    mem = fit_and_eval_probe(cal_df, test_df, x_best_cols, representation=f"z_combined_{x_best_name}")
    ent = eval_entropy_baseline(test_df)
    phase3_coefs[seed] = mem["coefficients"]

    delta_auroc = mem["auroc"] - ent["auroc"]
    delta_auprc = mem["auprc"] - ent["auprc"]

    for model, metrics in [
        ("entropy_H", ent),
        ("z_combined", {"auroc": mem["auroc"], "auprc": mem["auprc"], "n_test": mem["n_test"]}),
    ]:
        phase3_rows.append(
            {
                "seed": seed,
                "model": model,
                "representation": x_best_name,
                "n_cal": mem["n_cal"],
                "n_test": metrics["n_test"],
                "auroc": metrics["auroc"],
                "auprc": metrics["auprc"],
            }
        )

    phase3_rows.append(
        {
            "seed": seed,
            "model": "delta_z_combined_minus_H",
            "representation": x_best_name,
            "n_cal": mem["n_cal"],
            "n_test": mem["n_test"],
            "auroc": delta_auroc,
            "auprc": delta_auprc,
        }
    )

phase3_df = pd.DataFrame(phase3_rows)
pivot = phase3_df.pivot_table(index="seed", columns="model", values=["auroc", "auprc"], aggfunc="first")

z_auroc_mean = float(phase3_df.loc[phase3_df["model"] == "z_combined", "auroc"].mean())
h_auroc_mean = float(phase3_df.loc[phase3_df["model"] == "entropy_H", "auroc"].mean())
z_auprc_mean = float(phase3_df.loc[phase3_df["model"] == "z_combined", "auprc"].mean())
h_auprc_mean = float(phase3_df.loc[phase3_df["model"] == "entropy_H", "auprc"].mean())

phase3_summary = {
    "x_best": x_best_name,
    "memory_kind": "correctness_hd",
    "correctness_z_dim": CORRECTNESS_Z_DIM,
    "z_combined_dim": CORRECTNESS_Z_DIM * NUM_LEVELS,
    "mean_auroc_z_combined": z_auroc_mean,
    "mean_auroc_H": h_auroc_mean,
    "mean_delta_auroc": z_auroc_mean - h_auroc_mean,
    "mean_auprc_z_combined": z_auprc_mean,
    "mean_auprc_H": h_auprc_mean,
    "mean_delta_auprc": z_auprc_mean - h_auprc_mean,
    "z_combined_beats_H_on_mean_auroc": z_auroc_mean > h_auroc_mean,
    "test_sample_size": int(len(test_frames[SEEDS[0]])),
    "logistic_C": LOGISTIC_C,
}

results_payload = {
    "config": {
        "task": TASK,
        "seeds": list(SEEDS),
        "artifact_dir": str(ARTIFACT_DIR),
        "feature_cache_dir": str(FEATURE_CACHE_DIR),
        "memory_kind": "correctness_hd",
        "correctness_z_dim": CORRECTNESS_Z_DIM,
        "test_sample_size": TEST_SAMPLE_SIZE,
        "random_seed": RANDOM_SEED,
        "logistic_C": LOGISTIC_C,
    },
    "phase1_decision": phase1_decision,
    "phase1_per_seed": phase1_df.to_dict(orient="records"),
    "phase3_summary": phase3_summary,
    "phase3_per_seed": phase3_df.to_dict(orient="records"),
}
with open(PROBE_OUTPUT_DIR / "correctness_probe_results.json", "w", encoding="utf-8") as f:
    json.dump(results_payload, f, indent=2, default=str)

phase1_df.to_csv(PROBE_OUTPUT_DIR / "correctness_phase1_per_seed.csv", index=False)
phase3_df.to_csv(PROBE_OUTPUT_DIR / "correctness_phase3_per_seed.csv", index=False)

print("Phase 3 per-seed comparison")
display(phase3_df)
print("\nPhase 3 pivot (AUROC / AUPRC by seed)")
display(pivot)
print("\nPhase 3 summary:")
for k, v in phase3_summary.items():
    print(f"  {k}: {v}")

for seed in SEEDS:
    print(f"\n--- seed={seed} z_combined coefficients ({x_best_name}) ---")
    display(phase3_coefs[seed])

Phase 3 per-seed comparison


,seed,model,representation,n_cal,n_test,auroc,auprc
0,42,entropy_H,z_combined,1602,2000,0.651493,0.498329
1,42,z_combined,z_combined,1602,2000,0.765281,0.628731
2,42,delta_z_combined_minus_H,z_combined,1602,2000,0.113788,0.130402
3,123,entropy_H,z_combined,1602,2000,0.685894,0.521464
4,123,z_combined,z_combined,1602,2000,0.750404,0.611199
5,123,delta_z_combined_minus_H,z_combined,1602,2000,0.064509,0.089735
6,456,entropy_H,z_combined,1602,2000,0.656606,0.473923
7,456,z_combined,z_combined,1602,2000,0.744384,0.582077
8,456,delta_z_combined_minus_H,z_combined,1602,2000,0.087778,0.108154



Phase 3 pivot (AUROC / AUPRC by seed)


auprc                                         auroc  \
model delta_z_combined_minus_H entropy_H z_combined delta_z_combined_minus_H   
seed                                                                           
42                    0.130402  0.498329   0.628731                 0.113788   
123                   0.089735  0.521464   0.611199                 0.064509   
456                   0.108154  0.473923   0.582077                 0.087778   

                            
model entropy_H z_combined  
seed                        
42     0.651493   0.765281  
123    0.685894   0.750404  
456    0.656606   0.744384


Phase 3 summary:
  x_best: z_combined
  memory_kind: correctness_hd
  correctness_z_dim: 7
  z_combined_dim: 28
  mean_auroc_z_combined: 0.7533562491985651
  mean_auroc_H: 0.6646646307872054
  mean_delta_auroc: 0.08869161841135975
  mean_auprc_z_combined: 0.6073357589360162
  mean_auprc_H: 0.49790551211923884
  mean_delta_auprc: 0.10943024681677738
  z_combined_beats_H_on_mean_auroc: True
  test_sample_size: 2000
  logistic_C: 1.0

--- seed=42 z_combined coefficients (z_combined) ---


,feature,coef_standardized,coef_raw_scale
0,z1_d0,-0.315986,-7.180301e+02
1,z1_d1,1.082159,1.423543e+03
2,z1_d2,-1.618479,-1.993959e+03
3,z1_d3,-1.988990,-1.980375e+03
4,z1_d4,0.268582,5.671051e+02
5,z1_d5,0.164520,1.203149e+03
6,z1_d6,-1.232171,-8.746039e+03
7,z2_d0,-0.870484,-1.289470e+04
8,z2_d1,0.972995,6.338842e+03
9,z2_d2,-1.370615,-9.795633e+03



--- seed=123 z_combined coefficients (z_combined) ---


,feature,coef_standardized,coef_raw_scale
0,z1_d0,0.153416,224.020885
1,z1_d1,1.850349,2312.608742
2,z1_d2,-0.795427,-1175.742236
3,z1_d3,2.442849,4160.463577
4,z1_d4,-0.197260,-167.481171
5,z1_d5,0.123285,613.279311
6,z1_d6,1.525688,2663.812608
7,z2_d0,-0.333107,-2727.450094
8,z2_d1,0.410159,2174.073956
9,z2_d2,0.212327,1454.793453



--- seed=456 z_combined coefficients (z_combined) ---


,feature,coef_standardized,coef_raw_scale
0,z1_d0,1.147628,5526.091544
1,z1_d1,2.639270,4722.803148
2,z1_d2,0.824487,2335.418371
3,z1_d3,1.038601,2553.235151
4,z1_d4,-1.369591,-3923.535540
5,z1_d5,-0.006057,-21.210533
6,z1_d6,0.206287,816.860005
7,z2_d0,-0.127854,-2659.862959
8,z2_d1,1.126949,7686.196054
9,z2_d2,0.709605,7642.652681
